# Roberta

para este caso usaremos 

In [1]:
from deep_translator import GoogleTranslator  # Para traducción automática
from langdetect import detect  # Para detectar el idioma de un texto
from sentence_transformers import SentenceTransformer  # Para generar embeddings con RoBERTa

import torch  # Para usar la GPU si está disponible
import os  # Para manejar rutas de archivos

import faiss  # Para búsquedas rápidas en el índice de embeddings
import numpy as np
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(f"{Path.cwd()}").resolve()

def translate_to_english(text):
    """
    Traduce la consulta del usuario a inglés si no está en inglés.
    """
    try:
        detected_lang = detect(text)
        if detected_lang != 'en':
            translated_text = GoogleTranslator(source='auto', target='en').translate(text)
            print(f"Traducido: '{text}' → '{translated_text}'")
            return translated_text
        return text
    except Exception as e:
        print(f"Error al detectar idioma: {e}")
        return text

file_path = r"..\data\processed\df_clean.parquet"

df = pd.read_parquet(file_path)

data = ['name', 'release_date', 'required_age', 'price', 'dlc_count',
       'about_the_game', 'windows', 'mac', 'linux', 'metacritic_score',
       'achievements', 'recommendations', 'supported_languages',
       'full_audio_languages', 'developers', 'categories', 'genres', 'estimated_owners',
       'average_playtime_forever', 'median_playtime_forever', 'peak_ccu', 'tags']

# 📌 Crear un gran "combo textual" que contenga toda la información relevante
df['combined_text'] = (
    df['song_name'].astype(str).fillna('') + " " +
    df['artist_name'].astype(str).fillna('') + " " +
    df['cleaned_lyrics'].astype(str).fillna('') + " " +
    df['combined_genres'].astype(str).fillna('') + " " +
    df['playlists_names'].astype(str).fillna('') + " " +
    df['album_release_date'].astype(str).fillna('') + " " +
    df['language'].astype(str).fillna('')
)
for x in data:
    print(f"df[{x}]")

ModuleNotFoundError: No module named 'faiss'

In [ ]:

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando: {device.upper()}")

model = SentenceTransformer('https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2', device=device)

print("Generando nuevos embeddings (vectores de 1024 dimensiones)...")
embeddings_list = model.encode(df['combined_text'].tolist(), batch_size=32, show_progress_bar=True)

# 📌 Convertir embeddings a un array de NumPy en formato float32
embeddings_np = np.array(embeddings_list, dtype=np.float32)
d = embeddings_np.shape[1]  # Dimensión del vector (1024 en este caso)

# 📌 Crear el índice FAISS con `IndexFlatIP` (Producto Interno) sin normalización
index = faiss.IndexFlatIP(d)
index.add(embeddings_np)

# 📌 Guardar FAISS en archivo con manejo de errores
try:
    faiss.write_index(index, faiss_index_path)
    print(f"✅ FAISS `IndexFlatIP` guardado en {faiss_index_path}")
except Exception as e:
    print(f"❌ Error al guardar FAISS index: {e}")

# 📌 Guardar embeddings en Pickle para futuras consultas (🔥 SOLUCIONADO EL ERROR)
try:
    with open(embeddings_file, "wb") as f:
        pickle.dump(embeddings_list, f)  # Guardar correctamente como lista
    print(f"✅ Embeddings guardados en {embeddings_file}")
except Exception as e:
    print(f"❌ Error al guardar embeddings en Pickle: {e}")

In [ ]:
query = "quiero un juegos de acción y terror"
query_translated = translate_to_english(query)

# 📌 Generar el embedding de la consulta con RoBERTa
query_embedding = model.encode([query_translated], device=device, show_progress_bar=False)
query_embedding_np = np.array(query_embedding, dtype=np.float32)

# 📌 Realizar la búsqueda en FAISS (top 5 resultados)
k = 5
distances, indices = index.search(query_embedding_np, k)

# 📌 Mostrar resultados
if indices.size == 0:
    print("❌ No se encontraron resultados.")
else:
    results = []
    for i, idx in enumerate(indices[0]):
        sim_score = distances[0][i]
        song_info = df.iloc[idx].to_dict()
        song_info['similarity'] = sim_score
        results.append(song_info)

    results_df = pd.DataFrame(results)
    results_df
    #print(results_df[['song_name', 'artist_name', 'spotify_url', 'similarity']].to_string(index=False))